# Лабораторна робота 8. Перенесення навчання (Transfer Learning): VGG16

Ноутбук 1 з 3. Виконав: Бабин Богдан Олегович, група 443-4

**VGG16** — 16 шарів: 13 згорткових (фільтри 3×3) + 3 повнозв’язні; ~14,7 млн параметрів у згортковій частині. Мережа попередньо навчена на ImageNet (1,2 млн зображень, 1000 класів).
Ми беремо **лише згорткову частину** (`include_top=False`), **заморожуємо** її ваги і навчаємо поверх неї
власний класифікатор «кіт / собака».

**Перед запуском:** Среда выполнения → T4 GPU → «Выполнить все».

In [ ]:
import os, shutil, time, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, Flatten, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import load_img, img_to_array
from google.colab import files
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
%matplotlib inline

tf.keras.utils.set_random_seed(42)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Набір даних Dogs vs Cats
Набір з https://www.kaggle.com/c/dogs-vs-cats (25 000 фото). Kaggle вимагає акаунт, тому завантажуємо той самий набір
з офіційного дзеркала Microsoft.

In [ ]:
!wget -q -O cats_dogs.zip https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
!unzip -q -o cats_dogs.zip
!ls

## 2. Скрипт підготовки даних (з лабораторної 7)
Відкидає пошкоджені файли, перемішує і ділить на train / val / test (70 / 15 / 15).
Для перенесення навчання достатньо **3 000 зображень кожного класу** (train 4 200, val 900, test 900):
згорткова частина вже навчена на ImageNet, а навчати треба лише невеликий класифікатор, тому кількість експериментів можна збільшити.

In [ ]:
%%writefile prepare_dataset.py
"""
prepare_dataset.py — підготовка набору Dogs vs Cats для навчання нейромережі в Keras.

1) знаходить усі зображення котів і собак (підтримує обидва формати архіву:
   PetImages/Cat, PetImages/Dog  — дзеркало Microsoft,  або  train/cat.N.jpg, train/dog.N.jpg — Kaggle);
2) відкидає пошкоджені файли (у наборі є порожні та «биті» зображення);
3) перемішує і ділить на навчальний (70%), оціночний (15%) і тестовий (15%) набори;
4) копіює у структуру каталогів, яку розуміє ImageDataGenerator.flow_from_directory:
      dataset/train/cats, dataset/train/dogs, dataset/val/..., dataset/test/...
"""
import os, sys, shutil, random
from PIL import Image

dataset_dir = 'dataset'                         # куди складаємо результат
train_dir = os.path.join(dataset_dir, 'train')  # каталог з даними для навчання
val_dir = os.path.join(dataset_dir, 'val')      # каталог з даними для перевірки
test_dir = os.path.join(dataset_dir, 'test')    # каталог з даними для тестування
val_data_portion = 0.15                         # частка зображень для перевірки
test_data_portion = 0.15                        # частка зображень для тестування
random.seed(42)
# необов'язковий аргумент: максимум зображень кожного класу (для навчання без GPU), напр. python prepare_dataset.py 3000
max_per_class = int(sys.argv[1]) if len(sys.argv) > 1 and sys.argv[1].isdigit() else None


def find_images():
    """Повертає словник {'cats': [шляхи], 'dogs': [шляхи]}."""
    if os.path.isdir('PetImages'):              # дзеркало Microsoft (той самий набір Kaggle)
        lst = lambda d: [os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith('.jpg')]
        return {'cats': lst('PetImages/Cat'), 'dogs': lst('PetImages/Dog')}
    if os.path.isdir('train'):                  # оригінальний train.zip з Kaggle: cat.0.jpg, dog.0.jpg ...
        files = os.listdir('train')
        return {'cats': [os.path.join('train', f) for f in files if f.startswith('cat.')],
                'dogs': [os.path.join('train', f) for f in files if f.startswith('dog.')]}
    raise FileNotFoundError('Не знайдено ні PetImages/, ні train/ — розпакуйте архів з набором даних')


def is_valid(path):
    """Перевіряє, що файл — справжнє зображення, яке відкривається."""
    try:
        if os.path.getsize(path) == 0:
            return False
        with Image.open(path) as img:
            img.verify()                        # перевірка цілісності файлу
        with Image.open(path) as img:
            img.convert('RGB').load()           # повне декодування
        return True
    except Exception:
        return False


def create_directory(dir_name):
    """Створює каталог з підкаталогами cats і dogs (старий видаляє)."""
    if os.path.exists(dir_name):
        shutil.rmtree(dir_name)
    os.makedirs(os.path.join(dir_name, 'cats'))
    os.makedirs(os.path.join(dir_name, 'dogs'))


def copy_images(files, dest_dir, class_name):
    for i, src in enumerate(files):
        shutil.copy2(src, os.path.join(dest_dir, class_name, f'{class_name[:-1]}.{i}.jpg'))


images = find_images()
for cls in images:
    total = len(images[cls])
    images[cls] = [p for p in images[cls] if is_valid(p)]
    print(f'{cls}: знайдено {total}, пошкоджених відкинуто {total - len(images[cls])}')

# однакова кількість зображень кожного класу
nb_images = min(len(images['cats']), len(images['dogs']))
if max_per_class:
    nb_images = min(nb_images, max_per_class)
    print(f'Обмеження: {nb_images} зображень кожного класу')
start_val_data_idx = int(nb_images * (1 - val_data_portion - test_data_portion))
start_test_data_idx = int(nb_images * (1 - test_data_portion))

for d in (train_dir, val_dir, test_dir):
    create_directory(d)

for cls, files in images.items():
    random.shuffle(files)
    files = files[:nb_images]
    copy_images(files[:start_val_data_idx], train_dir, cls)
    copy_images(files[start_val_data_idx:start_test_data_idx], val_dir, cls)
    copy_images(files[start_test_data_idx:], test_dir, cls)

for d in (train_dir, val_dir, test_dir):
    print(f'{d}: cats = {len(os.listdir(os.path.join(d, "cats")))}, dogs = {len(os.listdir(os.path.join(d, "dogs")))}')

In [ ]:
!python prepare_dataset.py 3000

In [ ]:
for root, dirs, fs in sorted(os.walk('dataset')):
    depth = root.count(os.sep)
    print('    ' * depth + os.path.basename(root) + '/' + (f'   ({len(fs)} файлів)' if fs else ''))

## 3. Генератори
Замість `rescale=1/255` використовуємо **`preprocess_input` мережі VGG16**: зображення мають бути оброблені так само,
як під час навчання мережі на ImageNet, інакше перенесені ознаки працюють гірше.

In [ ]:
img_width, img_height = 150, 150
input_shape = (img_width, img_height, 3)
batch_size = 32

datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
train_generator = datagen.flow_from_directory('dataset/train', target_size=(img_width, img_height),
                                              batch_size=batch_size, class_mode='binary')
val_generator = datagen.flow_from_directory('dataset/val', target_size=(img_width, img_height),
                                            batch_size=batch_size, class_mode='binary')
test_generator = datagen.flow_from_directory('dataset/test', target_size=(img_width, img_height),
                                             batch_size=batch_size, class_mode='binary', shuffle=False)
print(train_generator.class_indices)

## 4. Завантажуємо попередньо навчену мережу VGG16
- `weights='imagenet'` — ваги, навчені на ImageNet;
- `include_top=False` — без повнозв’язних шарів класифікатора на 1000 класів (нам потрібні лише 2 класи);
- `input_shape=(150, 150, 3)` — розмір наших зображень.

In [ ]:
base_net = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
base_net.trainable = False      # заморожуємо всі шари: ваги ImageNet не змінюються під час навчання
print('Шарів у згортковій частині:', len(base_net.layers))
print('Вихід згорткової частини:', base_net.output_shape)
print(f'Параметрів: {base_net.count_params():,}  (навчаються: {sum(int(np.prod(w.shape)) for w in base_net.trainable_weights)})')

## 5. Складена мережа: VGG16 + власний класифікатор
- **Flatten** — як у лекції: карта ознак VGG16 4×4×512 → вектор 8192;
- **Dense(256, relu)** — новий повнозв’язний шар;
- **Dropout(d)** — гіперпараметр, який досліджуємо;
- **Dense(1, sigmoid)** — вихід: ймовірність класу «собака».

Оптимізатор Adam з малою швидкістю навчання `1e-4`, функція втрат `binary_crossentropy`.

In [ ]:
def build_model(dropout):
    model = Sequential([
        Input(shape=input_shape),
        base_net,
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(dropout),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=1e-4), metrics=['accuracy'])
    return model

m = build_model(0.5)
m.summary()
trainable = sum(int(np.prod(w.shape)) for w in m.trainable_weights)
print(f'Усього параметрів: {m.count_params():,} | навчаються: {trainable:,} | заморожені: {m.count_params() - trainable:,}')

## 6. Навчання і дослідження впливу Dropout
Навчаємо 4 моделі з Dropout = 0.0, 0.3, 0.5, 0.7 по **8 епох** без ранньої зупинки —
щоб на графіках було видно, як Dropout впливає на перенавчання (розрив між кривими навчання і перевірки).

In [ ]:
def plot_history(h, title):
    h = h.history; ep = range(1, len(h['loss']) + 1)
    plt.figure(figsize=(13, 4))
    plt.subplot(1, 2, 1); plt.plot(ep, h['accuracy'], 'b-o', ms=3, label='Навчальні'); plt.plot(ep, h['val_accuracy'], 'r-o', ms=3, label='Перевірочні')
    plt.title('Точність'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
    plt.subplot(1, 2, 2); plt.plot(ep, h['loss'], 'b-o', ms=3, label='Навчальні'); plt.plot(ep, h['val_loss'], 'r-o', ms=3, label='Перевірочні')
    plt.title('Помилка (loss)'); plt.xlabel('Епоха'); plt.legend(); plt.grid(alpha=.3)
    plt.suptitle(title); plt.tight_layout(); plt.show()

DROPOUTS = [0.0, 0.3, 0.5, 0.7]
EPOCHS = 8
results, histories, models = [], {}, {}
for d in DROPOUTS:
    print('=' * 70, f'\nDropout = {d}')
    tf.keras.utils.set_random_seed(42)
    model = build_model(d)
    start = time.time()
    h = model.fit(train_generator, epochs=EPOCHS, validation_data=val_generator, verbose=2)
    t = time.time() - start
    test_loss, test_acc = model.evaluate(test_generator, verbose=0)
    hh = h.history
    results.append({'Dropout': d,
                    'train acc, %': round(hh['accuracy'][-1] * 100, 2),
                    'val acc, %': round(hh['val_accuracy'][-1] * 100, 2),
                    'найкраща val acc, %': round(max(hh['val_accuracy']) * 100, 2),
                    'розрив train−val, %': round((hh['accuracy'][-1] - hh['val_accuracy'][-1]) * 100, 2),
                    'train loss': round(hh['loss'][-1], 4), 'val loss': round(hh['val_loss'][-1], 4),
                    'test acc, %': round(test_acc * 100, 2), 'test loss': round(test_loss, 4),
                    'час, с': round(t)})
    histories[d], models[d] = h, model
    plot_history(h, f'VGG16, Dropout = {d}: test acc {test_acc*100:.2f}%')

## 7. Порівняння: вплив Dropout на перенавчання

In [ ]:
res = pd.DataFrame(results)
res.insert(0, 'модель', 'VGG16')
res.to_csv('lab8_VGG16_dropout.csv', index=False)
res

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for d, h in histories.items():
    ax[0].plot(range(1, len(h.history['val_accuracy']) + 1), h.history['val_accuracy'], '-o', ms=3, label=f'Dropout {d}')
    ax[1].plot(range(1, len(h.history['accuracy']) + 1), np.array(h.history['accuracy']) - np.array(h.history['val_accuracy']), '-o', ms=3, label=f'Dropout {d}')
ax[0].set_title('Точність на перевірочних даних'); ax[0].set_xlabel('Епоха'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title('Розрив train acc − val acc (перенавчання)'); ax[1].set_xlabel('Епоха'); ax[1].axhline(0, color='k', lw=.5)
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 8. Найкраща модель: збереження

In [ ]:
best_d = res.loc[res['test acc, %'].idxmax(), 'Dropout']
best_model = models[best_d]
print('Найкраща модель: Dropout =', best_d, '| test acc =', res['test acc, %'].max(), '%')
best_model.save('lab8_VGG16_best.keras')
print('Розмір файлу:', round(os.path.getsize('lab8_VGG16_best.keras') / 2**20, 1), 'МБ')

## 9. Розпізнавання нових зображень
Ті самі власні зображення, що й у лабораторній 7 (20 звичайних і 6 «складних» — тварина в траві), щоб порівняти,
чи покращилось розпізнавання після перенесення навчання. Завантажте архів `lab8_test_images.zip`
(або окремі файли з назвами `cat_…` / `dog_…`).

In [ ]:
os.makedirs('test_images', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('test_images')
    else:
        shutil.move(name, os.path.join('test_images', name))
paths = sorted(os.path.join(r, f) for r, _, fs in os.walk('test_images') for f in fs
               if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith('._'))
print(len(paths), 'зображень')

In [ ]:
def predict_images(paths, title):
    rows = []
    cols = 6; rows_n = int(np.ceil(len(paths) / cols))
    plt.figure(figsize=(cols * 2.8, rows_n * 3.1))
    for i, p in enumerate(paths):
        x = img_to_array(load_img(p, target_size=(img_width, img_height)))
        x = preprocess_input(np.expand_dims(x, axis=0))
        prob = float(best_model.predict(x, verbose=0)[0][0])
        answer = 'собака' if prob > 0.5 else 'кіт'
        name = os.path.basename(p).lower()
        truth = 'кіт' if name.startswith('cat') else ('собака' if name.startswith('dog') else None)
        ok = None if truth is None else answer == truth
        conf = prob if prob > 0.5 else 1 - prob
        plt.subplot(rows_n, cols, i + 1); plt.imshow(load_img(p)); plt.axis('off')
        plt.title(f'{answer} ({conf*100:.0f}%)\n{os.path.basename(p)}', fontsize=8,
                  color='black' if ok is None else ('green' if ok else 'red'))
        rows.append({'файл': os.path.basename(p), 'справжній клас': truth or '?', 'відповідь': answer,
                     'впевненість, %': round(conf * 100, 1), 'вірно': ok})
    plt.suptitle(title); plt.tight_layout(); plt.show()
    df = pd.DataFrame(rows)
    k = df[df['вірно'].notna()]
    print(f'{title}: вірно {int(k["вірно"].sum())} з {len(k)}')
    return df

normal = [p for p in paths if '_h' not in os.path.basename(p)]
hard = [p for p in paths if '_h' in os.path.basename(p) and '_crop' not in os.path.basename(p)]
own = predict_images(normal, 'VGG16: звичайні зображення')
own_hard = predict_images(hard, 'VGG16: складні зображення (тварина в траві)') if hard else pd.DataFrame()
pd.concat([own, own_hard]).to_csv('lab8_VGG16_own.csv', index=False)
pd.concat([own, own_hard])

In [ ]:
summary = pd.DataFrame([{'модель': 'VGG16', 'найкращий Dropout': best_d,
    'test acc, %': res['test acc, %'].max(),
    'власні фото': f"{int(own['вірно'].sum())}/{len(own)}",
    'складні фото': f"{int(own_hard['вірно'].sum())}/{len(own_hard)}" if len(own_hard) else '—',
    'параметрів, млн': round(best_model.count_params() / 1e6, 1),
    'навчаються, млн': round(sum(int(np.prod(w.shape)) for w in best_model.trainable_weights) / 1e6, 2)}])
summary.to_csv('lab8_VGG16_summary.csv', index=False)
summary

In [ ]:
for f in ['lab8_VGG16_best.keras', 'lab8_VGG16_dropout.csv', 'lab8_VGG16_own.csv', 'lab8_VGG16_summary.csv']:
    files.download(f)